# Topic 6 · NumPy — Exercises

Same datasets as the demos (`Data/` folder). 

**Rules of this exercises**
- No Python `for` loops over data rows unless the task says so. Loops over a handful of items (3 species, 4 shops) are fine.
- Each task ends with a **check cell**. Do not edit it; it must run without error.
- Each task has an **explain-why** question. Answer in 1–2 sentences of your own words; these are graded.
- Before you submit: *Kernel → Restart & Run All*.

Full name: Vũ Duy Nhất

Student ID: 24127095

Class: 24KHDL

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)
np.set_printoptions(suppress=True, precision=4)

grades = np.genfromtxt("Data/P4DS-Grades.csv", delimiter=",", skip_header=1)
X = grades[:, :4]                                   # HW1, HW2, HW3, Project
official = grades[:, 4]
weights = np.array([0.5 / 3, 0.5 / 3, 0.5 / 3, 0.5])
final = X @ weights
grades.shape

(45, 5)

---
## A1. Drop the lowest homework

The lecturer considers a new policy: **drop each student's lowest homework**, and the remaining two homeworks count 25% each (project still 50%).

Here is a pure-Python version:

In [2]:
def final_drop_lowest_loop(X):
    out = []
    for hw1, hw2, hw3, project in X.tolist():
        best_two = sorted([hw1, hw2, hw3])[1:]
        out.append(0.25 * best_two[0] + 0.25 * best_two[1] + 0.5 * project)
    return np.array(out)

reference_A1 = final_drop_lowest_loop(X)
reference_A1[:5]

array([0.    , 9.75  , 8.3125, 8.9625, 9.4375])

**Task:**
1. Write `final_drop_lowest(X)` **without a loop** (hint: `np.sort` with the right `axis`).
2. Which student (row index) gains the most from the new policy, and by how much?
3. How many students change from **fail** to **pass** (pass mark 5)?

In [3]:
def final_drop_lowest(X):
    # YOUR CODE HERE
    project = X[:, -1].reshape(-1,1)
    best_two = np.sort(X[:, :3], axis=1)[:, 1:]
    new_X = np.concatenate([best_two, project], axis=1)
    weights = np.array([0.25, 0.25, 0.5])
    return new_X @ weights
    

new_final = final_drop_lowest(X)
best_gain_row = np.argmax(new_final - final)      # row index of the student who gains the most
best_gain = np.max(new_final - final)          # how many points they gain
n_fail_to_pass = np.sum((final < 5) & (new_final >= 5))

In [4]:
# CHECK (do not edit)
assert np.allclose(new_final, reference_A1)
assert np.isclose(best_gain, (new_final - final).max())
assert best_gain_row == np.argmax(new_final - final)
print("A1 OK")

A1 OK


**Explain why:** Why `axis=1` and not `axis=0` in `np.sort`? What would `axis=0` compute?

> *Your answer:* With axis=1, NumPy sorts the three homework scores within each student’s row, so we can drop that student’s lowest score. With axis=0, it sorts each homework column across students.

## A2. View or copy? Predict, then run

For each snippet, **first** write your prediction (does `a` change? yes/no), **then** run it. Use `np.shares_memory` to confirm.

| # | Snippet | `a` changes? (your prediction) |
|---|---|---|
| 1 | `b = a[1:4]; b[0] = -1` | Yes |
| 2 | `b = a[[1, 2, 3]]; b[0] = -1` | No |
| 3 | `b = a[a > 2]; b[0] = -1` | No |
| 4 | `b = a.reshape(2, 3); b[0, 0] = -1` | Yes |
| 5 | `b = a[::2]; b *= 10` | Yes |
| 6 | `b = a.copy()[1:4]; b[0] = -1` | No |

In [5]:
snippets = [
    "b = a[1:4]; b[0] = -1",
    "b = a[[1, 2, 3]]; b[0] = -1",
    "b = a[a > 2]; b[0] = -1",
    "b = a.reshape(2, 3); b[0, 0] = -1",
    "b = a[::2]; b *= 10",
    "b = a.copy()[1:4]; b[0] = -1",
]
for s in snippets:
    a = np.arange(6)
    original = a.copy()
    ns = {"a": a, "np": np}
    exec(s, ns)
    changed = not np.array_equal(ns["a"], original)
    print(f"{s:<36} a changed: {changed!s:<5}  shares memory: {np.shares_memory(ns['a'], ns['b'])}")

b = a[1:4]; b[0] = -1                a changed: True   shares memory: True
b = a[[1, 2, 3]]; b[0] = -1          a changed: False  shares memory: False
b = a[a > 2]; b[0] = -1              a changed: False  shares memory: False
b = a.reshape(2, 3); b[0, 0] = -1    a changed: True   shares memory: True
b = a[::2]; b *= 10                  a changed: True   shares memory: True
b = a.copy()[1:4]; b[0] = -1         a changed: False  shares memory: False


**Explain why:** Snippet 2 and snippet 1 select the same three elements. Why does only one of them change `a`?

> *Your answer:* Snippet 1 uses slicing, so b is a view that shares memory with a and changing b also changes a. Snippet 2 uses fancy indexing, so b is a copy and the assignment only affects b.

## A3. Questions for the grade report

Answer each with **one expression** (no loops). Use `final` (computed grades) unless stated.

1. `n_all_hw_9`: how many students scored **at least 9 on every** homework?
2. `n_improved`: how many students improved by **at least 1 point** from HW1 to HW3?
3. `n_project_carried`: how many students have project **≥ 8** but at least one homework **< 5**?
4. `hw_best_counts`: for each homework, how many students had it as their **best** homework (ties → the first one)? Result: an array of length 3.
5. `pct_letter`: percentage of students per letter grade A/B/C/D/F using the bands A ≥ 8.5, B ≥ 7, C ≥ 5.5, D ≥ 4, else F, as a dict `{"A": ..., ...}`.

In [6]:
hw = X[:, :3]
n_all_hw_9 = np.sum(np.all(hw >= 9, axis=1))
n_improved = np.sum((hw[:, 2] - hw[:, 0]) >= 1)
n_project_carried = np.sum(((X[:, 3]) >= 8) & (np.any(hw < 5, axis=1)))
hw_best_counts = np.bincount(np.argmax(hw, axis=1), minlength=3)
pct_letter = dict(zip(["A", "B", "C", "D", "F"], 
             np.bincount(np.searchsorted(["A", "B", "C", "D", "F"], 
             np.select([final >= 8.5, final >= 7, final >= 5.5, final >= 4 ],
             ["A", "B", "C", "D"], default="F")), minlength=5) * 100 / final.size))

In [7]:
# CHECK (do not edit)
assert int(n_all_hw_9) == sum(all(v >= 9 for v in row[:3]) for row in X.tolist())
assert int(n_improved) == sum(row[2] - row[0] >= 1 for row in X.tolist())
assert np.asarray(hw_best_counts).sum() == len(X)
assert np.isclose(sum(pct_letter.values()), 100)
print("A3 OK")

A3 OK


**Explain why:** In question 4, many students have three homeworks of 10. Which homework does `argmax` pick for them, and does that make the answer misleading?

> *Your answer:* When all three homework scores are 10, argmax picks HW1 because it appears first. The count follows the tie-breaking rule, but HW1 gets credit even when it shares the highest score with HW2 and HW3.

## A4. Measure performance

A university-wide version of the grade file has **1 million** students (simulated below).

1. The provided loop counts the students whose final grade is above the overall mean. Write `count_above_mean_np`, which does the same with no loop.
2. Time both with `%timeit` and report the speed-up.
3. Time `np.append` growing an array of 20,000 grades one by one vs. building a list and converting once.

In [8]:
big_X = np.clip(rng.normal(8, 1.5, (1_000_000, 4)), 0, 10).round(2)
big_final = big_X @ weights

def count_above_mean_loop(values):
    total = 0.0
    for v in values:
        total += v
    mean = total / len(values)
    count = 0
    for v in values:
        if v > mean:
            count += 1
    return count

In [9]:
def count_above_mean_np(values):
    # YOUR CODE HERE
    return np.sum(values > np.mean(values))

# time both versions with %timeit
loop_time = %timeit -o count_above_mean_loop(big_final)
np_time = %timeit -o count_above_mean_np(big_final)

speedup = loop_time.average / np_time.average
print(f"NumPy nhanh gấp {speedup:.2f} lần")

89.4 ms ± 1.03 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
700 μs ± 37 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
NumPy nhanh gấp 127.71 lần


In [10]:
%%timeit
# np.append 
np_arr = np.array([])
for i in range(20000):
    np_arr = np.append(np_arr, big_final[i])

58.2 ms ± 600 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [11]:
%%timeit 
# list.append
list_arr = []
for i in range(20000):
    list_arr.append(big_final[i])
np_convert = np.array(list_arr)

1.23 ms ± 11.4 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [12]:
# CHECK (do not edit)
assert count_above_mean_np(big_final[:200_000]) == count_above_mean_loop(big_final[:200_000])
print("A4 OK")

A4 OK


**Explain why:** `np.append` in a loop got slower *per element* as the array grew. Why?

> *Your answer:* Each call to `np.append` creates a new array and copies all the existing elements, so later appends take longer and the total cost is `O(n²)`. A Python list usually has spare capacity for new elements, so building the list and converting it to a NumPy array once takes `O(n)` overall.

---
**Before submitting:** *Kernel → Restart & Run All*, check that every `CHECK` cell prints OK, and that every explain-why answer is filled in.